In [1]:
# ==========================================
# BLOCK 1: ARCHITECTURE (CROSS-MODAL TEMPORAL)
# ==========================================
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import mlflow

# --- CONFIGURATION ---
BATCH_SIZE = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EXPERIMENT_NAME = "Teams_ActiveSpeaker_Temporal"
HARD_NEG_PROB = 0.1
mlflow.set_experiment(EXPERIMENT_NAME)

class AVAVectorDataset(Dataset):
    def __init__(self, root_dir, file_list, hard_negative_prob=0.0, pool_size=500):
        self.root_dir = root_dir
        self.file_list = file_list
        self.hard_negative_prob = hard_negative_prob
        self.audio_pool = []

        if self.hard_negative_prob > 0.0:
            print(f"Building Audio Pool (Size: {pool_size}) for Hard Negatives...")
            sample_list = file_list.copy()
            random.shuffle(sample_list)
            
            for f in sample_list:
                if len(self.audio_pool) >= pool_size:
                    break
                    
                data = torch.load(os.path.join(self.root_dir, f), weights_only=True)
                    
                if data['label'] == 1:
                    self.audio_pool.append(data['audio'])
            
            print(f"Audio Pool ready with {len(self.audio_pool)} active samples.")

    def __len__(self): 
        return len(self.file_list)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, self.file_list[idx])
        
        data = torch.load(path, weights_only=True)
        v, a, l = data['visual'], data['audio'], data['label']

        # Hard Negative Dubbing Check
        if self.hard_negative_prob > 0.0 and l == 0:
            if random.random() < self.hard_negative_prob and self.audio_pool:
                a = random.choice(self.audio_pool)

        return v, a, torch.tensor(l, dtype=torch.long)

class CrossAttentionActiveSpeakerModel(nn.Module):
    def __init__(self, dropout_audio=0.2, hidden_size=128):
        super(CrossAttentionActiveSpeakerModel, self).__init__()
        
        self.audio_conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.audio_dropout = nn.Dropout(dropout_audio)
        
        self.visual_proj = nn.Linear(512, hidden_size)
        self.audio_proj = nn.Linear(32, hidden_size)
        
        self.cross_attn = nn.MultiheadAttention(embed_dim=hidden_size, num_heads=4, batch_first=True)
        
        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size, 
            num_layers=1, 
            batch_first=True
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, v, a):
        B, T, V_dim = v.size()
        _, _, C, H, W = a.size()
        
        a_flat = a.view(B * T, C, H, W) 
        
        # Audio extraction and standard dropout
        a_feat = self.audio_conv(a_flat).view(B * T, -1) 
        a_feat = self.audio_dropout(a_feat)
        a_feat = a_feat.view(B, T, -1) 
        
        v_proj = self.visual_proj(v) # Shape: (B, 15, 128)
        a_proj = self.audio_proj(a_feat) # Shape: (B, 15, 128)
        
        attn_out, _ = self.cross_attn(query=v_proj, key=a_proj, value=a_proj)
        gru_out, _ = self.gru(attn_out)
        final_feat = gru_out.mean(dim=1) 
        
        main_out = self.classifier(final_feat)
        
        if self.training:
            # Return the mean of the projections to apply Contrastive Loss
            return main_out, v_proj.mean(dim=1), a_proj.mean(dim=1)
            
        return main_out

print(f"✅ Architecture defined on {DEVICE}")

/home/alpine/miniconda3/envs/dl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/02/25 17:13:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/25 17:13:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/25 17:13:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/25 17:13:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/25 17:13:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/25 17:13:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/25 17:13:41 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/25 17:13:41 INFO alembic.runtime.migration: Will assume non-transactional DDL.


✅ Architecture defined on cuda


In [2]:
# ==========================================
# BLOCK 2: STRICT TRAIN/VAL + TEST LOADING
# ==========================================
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

print("Loading Data: 85/15 Train/Val Split + Locked Test Set...")

# IMPORTANT: Pointing to the NEW Temporal directories!
TRAIN_DIR = "./Data/Processed_Tensors_Temporal/Train/"
TEST_DIR = "./Data/Processed_Tensors_Temporal/Test/"

train_files_raw = [f for f in os.listdir(TRAIN_DIR) if f.endswith('.pt')]
test_files = [f for f in os.listdir(TEST_DIR) if f.endswith('.pt')]

if not train_files_raw: 
    raise ValueError("Missing .pt files in Temporal Train directory!")

train_val_video_ids = list(set([f.split('#')[0] for f in train_files_raw if '#' in f]))
random.seed(42) 
random.shuffle(train_val_video_ids)

num_val = max(1, int(len(train_val_video_ids) * 0.15))
val_vids = train_val_video_ids[:num_val]
train_vids = train_val_video_ids[num_val:]

train_files = [f for f in train_files_raw if f.split('#')[0] in train_vids]
val_files = [f for f in train_files_raw if f.split('#')[0] in val_vids]

train_ds = AVAVectorDataset(TRAIN_DIR, train_files, hard_negative_prob=HARD_NEG_PROB, pool_size=500)
val_ds = AVAVectorDataset(TRAIN_DIR, val_files, hard_negative_prob=0.0)
test_ds = AVAVectorDataset(TEST_DIR, test_files, hard_negative_prob=0.0)

WORKERS = 6 

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True)

print(f"Optimized Loaders Ready ({WORKERS} Workers).")
print(f"    TRAIN: {len(train_ds)} samples")
print(f"    VAL:   {len(val_ds)} samples")
print(f"    TEST:  {len(test_ds)} samples")

print("Initializing Model & Optimizer Globals...")
model = CrossAttentionActiveSpeakerModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.1, verbose=True)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
global_step = 0
best_val_acc = 0.0

print("Ready to start training session.")

Loading Data: 85/15 Train/Val Split + Locked Test Set...
Building Audio Pool (Size: 500) for Hard Negatives...
Audio Pool ready with 500 active samples.
Optimized Loaders Ready (6 Workers).
    TRAIN: 389203 samples
    VAL:   56311 samples
    TEST:  0 samples
Initializing Model & Optimizer Globals...
Ready to start training session.


/home/alpine/miniconda3/envs/dl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [3]:
def run_training(epochs=10):
    global global_step, global_epoch, best_val_acc, history, model, optimizer, scheduler, criterion
    
    contrastive_criterion = nn.CosineEmbeddingLoss(margin=0.5).to(DEVICE)
    CONTRASTIVE_WEIGHT = 0.1 
    
    for _ in range(epochs):
        global_epoch += 1
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        
        loop = tqdm(train_loader, desc=f"Epoch {global_epoch} [Train]")
        for v, a, l in loop:
            v, a, l = v.to(DEVICE), a.to(DEVICE), l.to(DEVICE)
            
            # Ignore the impossible auxiliary output
            main_out, v_embed, a_embed = model(v, a)
            
            loss_main = criterion(main_out, l)
            
            contrastive_target = torch.where(l == 1, torch.tensor(1.0).to(DEVICE), torch.tensor(-1.0).to(DEVICE))
            loss_contrast = contrastive_criterion(v_embed, a_embed, contrastive_target)
            
            # Clean, dual-objective loss
            loss = loss_main + (CONTRASTIVE_WEIGHT * loss_contrast)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            run_loss += loss.item()
            _, pred = torch.max(main_out, 1)
            correct += (pred == l).sum().item()
            total += l.size(0)
            
            global_step += 1
            if global_step % 20 == 0:
                mlflow.log_metric("train_loss_step", loss.item(), step=global_step)
            
            loop.set_postfix(loss=loss.item())

        torch.cuda.empty_cache()

        train_acc = 100. * correct / total
        train_loss = run_loss / len(train_loader)
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        
        with torch.no_grad():
            for v, a, l in tqdm(val_loader, desc=f"Epoch {global_epoch} [Val]"):
                v, a, l = v.to(DEVICE), a.to(DEVICE), l.to(DEVICE)
                out = model(v, a)
                loss = criterion(out, l)
                val_loss += loss.item()
                _, pred = torch.max(out, 1)
                val_correct += (pred == l).sum().item()
                val_total += l.size(0)

        val_acc = 100. * val_correct / val_total
        val_loss /= len(val_loader)
        
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss) 
        mlflow.log_metric("learning_rate", current_lr, step=global_epoch) 

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        }, step=global_epoch)
        
        print(f"Epoch {global_epoch} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | LR: {current_lr}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print(">>> Saved New Best Model!")

def start_session(run_name=None, description=None, learning_rate=5e-5, weight_decay=1e-4, 
                  hidden_size=128, label_smoothing=0.0, speaker_weight=2.0, contrastive_weight=0.1):
    
    global global_step, global_epoch, history, best_val_acc, model, optimizer, scheduler, criterion
    
    if mlflow.active_run(): mlflow.end_run()
    mlflow.start_run(run_name=run_name)
    
    global_step, global_epoch, best_val_acc = 0, 0, 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    if description: mlflow.set_tag("mlflow.note.content", description)

    model = CrossAttentionActiveSpeakerModel(hidden_size=hidden_size).to(DEVICE)
    
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    weights = torch.tensor([1.0, speaker_weight]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.1)

    mlflow.log_params({
        "batch_size": BATCH_SIZE,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "hidden_size": hidden_size,
        "dropout_input": 0.0, 
        "dropout_fusion": 0.3,
        "label_smoothing": label_smoothing,
        "scheduler": "ReduceLROnPlateau",
        "model_type": "CrossAttention_GRU_MeanPool_Contrastive_NoAux",
        "train_dataset_size": len(train_ds),
        "validation_dataset_size": len(val_ds),
        "test_dataset_size": len(test_ds),
        "auxiliary_loss_weight": 0.0,
        "speaker_weight": speaker_weight,
        "contrastive_loss_weight": contrastive_weight
    })
    print(f"Session Started: {run_name} | Removed Aux Loss")


In [4]:
start_session(
    run_name="Run_28C_Contrastive_NoAux_No_Rolling_fake_Audio",
    description="Removed rolling fake audio and zeroing out audio at 50% to see if that is causing the jitterning ofthe loss step function",
    learning_rate=5e-5,
    weight_decay=1e-4,
    hidden_size=128,
    contrastive_weight=0.1
)

run_training(3)

Session Started: Run_28C_Contrastive_NoAux_No_Rolling_fake_Audio | Removed Aux Loss


Epoch 1 [Val]: 100%|██████████████████████████████████████████████████████████████████| 220/220 [03:28<00:00,  1.06it/s]


Epoch 1 | Train Loss: 0.6020 Acc: 66.65% | Val Loss: 0.5508 Acc: 73.74% | LR: 5e-05
>>> Saved New Best Model!


Epoch 2 [Val]: 100%|██████████████████████████████████████████████████████████████████| 220/220 [03:49<00:00,  1.05s/it]


Epoch 2 | Train Loss: 0.5502 Acc: 72.97% | Val Loss: 0.5519 Acc: 76.30% | LR: 5e-05
>>> Saved New Best Model!


Epoch 3 [Val]: 100%|██████████████████████████████████████████████████████████████████| 220/220 [03:50<00:00,  1.05s/it]


Epoch 3 | Train Loss: 0.5322 Acc: 73.96% | Val Loss: 0.5179 Acc: 76.34% | LR: 5e-05
>>> Saved New Best Model!


In [ ]:
run_training(7)

Epoch 4 [Val]: 100%|██████████████████████████████████████████████████████████████████| 220/220 [03:44<00:00,  1.02s/it]


Epoch 4 | Train Loss: 0.5195 Acc: 74.79% | Val Loss: 0.4976 Acc: 74.23% | LR: 5e-05


Epoch 5 [Val]: 100%|██████████████████████████████████████████████████████████████████| 220/220 [03:42<00:00,  1.01s/it]


Epoch 5 | Train Loss: 0.5118 Acc: 75.21% | Val Loss: 0.5036 Acc: 75.99% | LR: 5e-05


Epoch 6 [Train]:   1%|▋                                                   | 21/1521 [00:16<15:49,  1.58it/s, loss=0.536]